# 04 — Preprocessing and Feature Engineering

## Smart India Real Estate Analytics

### Objective

The objective of this notebook is to transform the validated real-estate data into a clean, model-ready representation while preventing target leakage.

The preprocessing pipeline will include:

- parsing the `Price` target into a numeric representation
- selecting appropriate predictive features
- extracting structured information from `Property Title`
- handling categorical variables
- handling numerical variables
- excluding leakage-prone and unsuitable fields
- creating a reproducible preprocessing pipeline

### Feature Decisions from EDA

| Column | Treatment |
|---|---|
| `Price` | Target; convert to numeric |
| `Total_Area` | Numerical feature |
| `Baths` | Numerical feature |
| `Balcony` | Categorical feature |
| `Location` | Requires careful feature engineering |
| `Property Title` | Extract structured property information |
| `Name` | Exclude |
| `Description` | Exclude due to target leakage |
| `Price_per_SQFT` | Exclude due to target leakage |

### Data Integrity Rules

- The raw CSV will not be modified.
- Target values will not be artificially changed.
- `Price_per_SQFT` will not be used as a predictor.
- `Description` will not be used as a raw predictive feature.
- Preprocessing must be reproducible.
- Learned preprocessing operations must be fitted only on training data.
- The final preprocessing design must be compatible with the model-training and Streamlit stages.

In [30]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
TEST_SIZE = 0.20

In [31]:
DATA_PATH = Path("../data/raw/Real Estate Data V21.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df_raw.shape}")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

Dataset shape: (14528, 9)
Rows: 14,528
Columns: 9


In [32]:
def parse_price(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().replace("₹", "").replace(",", "").lower()

    try:
        if "cr" in value:
            return float(value.replace("cr", "").strip()) * 10_000_000

        elif "lacs" in value:
            return float(value.replace("lacs", "").strip()) * 100_000

        elif "lac" in value:
            return float(value.replace("lac", "").strip()) * 100_000

        elif "k" in value:
            return float(value.replace("k", "").strip()) * 1_000

        elif "l" in value:
            return float(value.replace("l", "").strip()) * 100_000

        else:
            return np.nan

    except ValueError:
        return np.nan


df_processed = df_raw.copy()

df_processed["Price_numeric"] = df_processed["Price"].apply(parse_price)

print(f"Valid numeric prices: {df_processed['Price_numeric'].notna().sum():,}")
print(f"Ambiguous/unparsed prices: {df_processed['Price_numeric'].isna().sum():,}")

Valid numeric prices: 14,525
Ambiguous/unparsed prices: 3


In [33]:
model_data = df_processed.dropna(
    subset=["Price_numeric"]
).copy()

print(f"Modeling dataset shape: {model_data.shape}")
print(f"Rows excluded due to ambiguous Price: {len(df_processed) - len(model_data):,}")

Modeling dataset shape: (14525, 10)
Rows excluded due to ambiguous Price: 3


In [34]:
X = model_data.drop(columns=["Price", "Price_numeric"])
y = model_data["Price_numeric"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
print(f"Training target: {y_train.shape}")
print(f"Test target: {y_test.shape}")

Training features: (11620, 8)
Test features: (2905, 8)
Training target: (11620,)
Test target: (2905,)


## Feature Selection and Engineering Plan

Based on data validation and EDA, the initial modeling feature set will use only information that is available before the property price is predicted.

### Excluded Features

- `Name` — excluded because of high cardinality and substantial overlap with `Location`.
- `Description` — excluded because the text explicitly contains property price and other target-related information, creating a leakage risk.
- `Price_per_SQFT` — excluded because it is potentially derived from `Price` and `Total_Area`, creating a target-leakage risk.

### Features Requiring Engineering

- `Property Title` — structured patterns such as BHK, RK, and Studio will be extracted rather than using the raw text directly.
- `Location` — high-cardinality categorical information requires a carefully selected representation.

### Direct Features

- `Total_Area` — numerical feature.
- `Baths` — numerical feature.
- `Balcony` — categorical feature.

The final feature set will be constructed after the engineered features have been created and validated.

In [35]:
feature_plan = pd.DataFrame({
    "Feature": [
        "Name",
        "Property Title",
        "Location",
        "Total_Area",
        "Price_per_SQFT",
        "Description",
        "Baths",
        "Balcony"
    ],
    "Initial Treatment": [
        "Exclude",
        "Engineer",
        "Engineer",
        "Keep",
        "Exclude",
        "Exclude",
        "Keep",
        "Keep"
    ]
})

feature_plan

,Feature,Initial Treatment
0,Name,Exclude
1,Property Title,Engineer
2,Location,Engineer
3,Total_Area,Keep
4,Price_per_SQFT,Exclude
5,Description,Exclude
6,Baths,Keep
7,Balcony,Keep


In [36]:
title_features = pd.DataFrame(index=model_data.index)

title_text = model_data["Property Title"].astype(str)

# BHK extraction.
# Handles both "2 BHK" and "5+ BHK".
bhk_extract_pattern = r"(?i)(\d+)\s*\+?\s*BHK"

title_features["BHK"] = pd.to_numeric(
    title_text.str.extract(
        bhk_extract_pattern,
        expand=False
    ),
    errors="coerce"
)

# Identify titles explicitly written as "5+ BHK".
five_plus_bhk_pattern = r"(?i)\d+\s*\+\s*BHK"

title_features["BHK_5_PLUS"] = (
    title_text.str.contains(
        five_plus_bhk_pattern,
        regex=True,
        na=False
    )
    .astype(int)
)

# Property configuration category.
title_features["Configuration"] = np.select(
    [
        title_text.str.contains(
            r"(?i)\d+\s*\+?\s*BHK",
            regex=True,
            na=False
        ),
        title_text.str.contains(
            r"(?i)\b\d+\s*RK\b",
            regex=True,
            na=False
        ),
        title_text.str.contains(
            r"(?i)\bStudio\b",
            regex=True,
            na=False
        )
    ],
    [
        "BHK",
        "RK",
        "Studio"
    ],
    default="Unknown"
)

title_features.head()

,BHK,BHK_5_PLUS,Configuration
0,4.0,0,BHK
1,10.0,0,BHK
2,3.0,0,BHK
3,7.0,0,BHK
4,2.0,0,BHK


In [37]:
print("BHK values:")
print(title_features["BHK"].value_counts(dropna=False).sort_index())

print("\nConfiguration counts:")
print(title_features["Configuration"].value_counts())

BHK values:
BHK
1.0     2586
2.0     5677
3.0     3121
4.0      984
5.0      616
6.0      295
7.0      143
8.0      132
9.0       78
10.0     166
NaN      727
Name: count, dtype: int64

Configuration counts:
Configuration
BHK        13798
RK           706
Studio        11
Unknown       10
Name: count, dtype: int64


In [38]:
original_bhk_pattern = r"\b\d+\s*BHK\b"
new_bhk_pattern = r"(?i)(\d+)\s*\+?\s*BHK"

original_bhk_mask = title_text.str.contains(
    original_bhk_pattern,
    case=False,
    regex=True,
    na=False
)

new_bhk_mask = title_text.str.contains(
    new_bhk_pattern,
    regex=True,
    na=False
)

new_matches = title_text[
    new_bhk_mask & ~original_bhk_mask
]

print(f"Original BHK matches: {original_bhk_mask.sum():,}")
print(f"New BHK matches: {new_bhk_mask.sum():,}")
print(f"Additional matches: {len(new_matches):,}")

print("\nAdditional matches:")
print(new_matches.to_list())

Original BHK matches: 13,774
New BHK matches: 13,798
Additional matches: 24

Additional matches:
['5+ BHK Independent House for sale in Avadi, Chennai', '5+ BHK Independent House for sale in Avadi, Chennai', '5+ BHK Independent House for sale in Kithaganur Colony, Bangalore', '5+ BHK Independent House for sale in RR Nagar, Bangalore', '5+ BHK Independent House for sale in Totagere, Bangalore', '5+ BHK Independent House for sale in Shivaji Nagar, Bangalore', '5+ BHK Independent House for sale in Bannerughatta, Bangalore', '5+ BHK Independent House for sale in Ramamurthy Nagar, Bangalore', '5+ BHK Independent House for sale in Electronic City Phase II, Bangalore', '5+ BHK Independent House for sale in Nagasandra, Bangalore', '5+ BHK Independent House for sale in R. T. Nagar, Bangalore', '5+ BHK Independent House for sale in Horamavu, Bangalore', '5+ BHK Independent House for sale in Ejipura, Bangalore', '5+ BHK Independent House for sale in Paikpara, Kolkata', '5+ BHK Independent House f

C:\Users\artis\AppData\Local\Temp\ipykernel_21224\2644371838.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  new_bhk_mask = title_text.str.contains(


In [39]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["BHK"] = title_features.loc[X_train.index, "BHK"]
X_train["BHK_5_PLUS"] = title_features.loc[X_train.index, "BHK_5_PLUS"]
X_train["Configuration"] = title_features.loc[X_train.index, "Configuration"]

X_test["BHK"] = title_features.loc[X_test.index, "BHK"]
X_test["BHK_5_PLUS"] = title_features.loc[X_test.index, "BHK_5_PLUS"]
X_test["Configuration"] = title_features.loc[X_test.index, "Configuration"]

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (11620, 11)
Test shape: (2905, 11)


In [40]:
X_train = X_train.drop(
    columns=["Name", "Property Title", "Price_per_SQFT", "Description"]
)

X_test = X_test.drop(
    columns=["Name", "Property Title", "Price_per_SQFT", "Description"]
)

print("Final training feature columns:")
print(X_train.columns.tolist())

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

Final training feature columns:
['Location', 'Total_Area', 'Baths', 'Balcony', 'BHK', 'BHK_5_PLUS', 'Configuration']

Training shape: (11620, 7)
Test shape: (2905, 7)


In [41]:
location_counts = X_train["Location"].value_counts()

print(f"Unique training locations: {location_counts.size:,}")
print(f"Locations appearing once: {(location_counts == 1).sum():,}")
print(f"Locations appearing more than once: {(location_counts > 1).sum():,}")

location_counts.head(20)

Unique training locations: 6,025
Locations appearing once: 4,306
Locations appearing more than once: 1,719


Location
Sector 12 Dwarka, New Delhi                           39
Bannerughatta, Bangalore                              30
Mattanahalli, Bangalore                               27
Wagholi, Pune                                         27
Chakan, Pune                                          26
Horamavu Agara, Horamavu,Bangalore                    24
Avadi, Chennai                                        23
Veer Sandra, Electronic City,Bangalore                23
Hindustan Antibiotics Colony, Pimpri,Pune             23
7th Phase, JP Nagar,Bangalore                         22
Narhe, Pune                                           22
Veer Sandra, Electronic City Phase II,Bangalore       21
Yelahanka, Bangalore                                  21
Dodsworth Layout, Whitefield,Bangalore                21
Sarjapur, Bangalore                                   21
Srinivasa Nagar, Banashankari,Bangalore               21
Bommasandra Industrial Area, Bommasandra,Bangalore    20
Nelamangala, Bangalore

In [42]:
location_text = X_train["Location"].astype(str).str.strip()

candidate_city = (
    location_text
    .str.split(",")
    .str[-1]
    .str.strip()
)

print(f"Unique candidate cities: {candidate_city.nunique():,}")

candidate_city.value_counts().head(20)

Unique candidate cities: 8


Location
Bangalore    3612
Pune         2382
New Delhi    1759
Chennai      1267
Mumbai       1086
Kolkata      1084
Hyderabad     426
Thane           4
Name: count, dtype: int64

In [43]:
city_price_analysis = pd.DataFrame({
    "City": candidate_city,
    "Price": y_train
})

city_price_summary = (
    city_price_analysis
    .groupby("City")["Price"]
    .agg(
        Property_Count="count",
        Median_Price="median",
        Mean_Price="mean"
    )
    .sort_values("Median_Price", ascending=False)
)

city_price_summary

,Property_Count,Median_Price,Mean_Price
City,,,
Mumbai,1086,9000000.0,1.497019e+07
Hyderabad,426,8500000.0,1.283672e+07
Bangalore,3612,7750000.0,1.198662e+07
Thane,4,6650000.0,6.662500e+06
New Delhi,1759,6000000.0,1.310904e+07
Chennai,1267,5850000.0,9.420081e+06
Pune,2382,4500000.0,6.821141e+06
Kolkata,1084,4200000.0,6.639010e+06


In [44]:
X_train["City"] = (
    X_train["Location"]
    .astype(str)
    .str.split(",")
    .str[-1]
    .str.strip()
)

X_test["City"] = (
    X_test["Location"]
    .astype(str)
    .str.split(",")
    .str[-1]
    .str.strip()
)

print("Training cities:")
print(X_train["City"].value_counts())

print("\nTest cities:")
print(X_test["City"].value_counts())

Training cities:
City
Bangalore    3612
Pune         2382
New Delhi    1759
Chennai      1267
Mumbai       1086
Kolkata      1084
Hyderabad     426
Thane           4
Name: count, dtype: int64

Test cities:
City
Bangalore    899
Pune         582
New Delhi    406
Chennai      328
Kolkata      308
Mumbai       266
Hyderabad    114
Thane          2
Name: count, dtype: int64


In [45]:
for threshold in [2, 3, 5, 10, 20]:
    frequent_locations = (
        location_counts[location_counts >= threshold]
        .index
    )

    print(
        f"Minimum frequency {threshold:>2}: "
        f"{len(frequent_locations):,} locations retained"
    )

Minimum frequency  2: 1,719 locations retained
Minimum frequency  3: 1,027 locations retained
Minimum frequency  5: 496 locations retained
Minimum frequency 10: 128 locations retained
Minimum frequency 20: 17 locations retained


In [46]:
LOCATION_MIN_FREQUENCY = 2

frequent_locations = set(
    location_counts[
        location_counts >= LOCATION_MIN_FREQUENCY
    ].index
)

X_train["Location_Grouped"] = X_train["Location"].where(
    X_train["Location"].isin(frequent_locations),
    "Other"
)

X_test["Location_Grouped"] = X_test["Location"].where(
    X_test["Location"].isin(frequent_locations),
    "Other"
)

print(
    f"Frequent locations retained: "
    f"{len(frequent_locations):,}"
)

print(
    f"Training grouped categories: "
    f"{X_train['Location_Grouped'].nunique():,}"
)

print(
    f"Test grouped categories: "
    f"{X_test['Location_Grouped'].nunique():,}"
)

Frequent locations retained: 1,719
Training grouped categories: 1,720
Test grouped categories: 819


In [47]:
train_other_count = (
    X_train["Location_Grouped"] == "Other"
).sum()

test_other_count = (
    X_test["Location_Grouped"] == "Other"
).sum()

print(f"Training records grouped as Other: {train_other_count:,}")
print(f"Test records grouped as Other: {test_other_count:,}")

print(
    f"Training Other percentage: "
    f"{train_other_count / len(X_train) * 100:.2f}%"
)

print(
    f"Test Other percentage: "
    f"{test_other_count / len(X_test) * 100:.2f}%"
)

Training records grouped as Other: 4,306
Test records grouped as Other: 1,403
Training Other percentage: 37.06%
Test Other percentage: 48.30%


In [48]:
threshold_analysis = []

for threshold in [2, 3, 5, 10, 20]:
    frequent_locations = location_counts[
        location_counts >= threshold
    ].index

    retained_count = X_train["Location"].isin(
        frequent_locations
    ).sum()

    other_count = len(X_train) - retained_count

    threshold_analysis.append({
        "Minimum_Frequency": threshold,
        "Locations_Retained": len(frequent_locations),
        "Properties_Retained": retained_count,
        "Properties_Retained_%": round(
            retained_count / len(X_train) * 100, 2
        ),
        "Properties_Other": other_count,
        "Other_%": round(
            other_count / len(X_train) * 100, 2
        )
    })

threshold_analysis_df = pd.DataFrame(threshold_analysis)

threshold_analysis_df

,Minimum_Frequency,Locations_Retained,Properties_Retained,Properties_Retained_%,Properties_Other,Other_%
0,2,1719,7314,62.94,4306,37.06
1,3,1027,5930,51.03,5690,48.97
2,5,496,4145,35.67,7475,64.33
3,10,128,1804,15.52,9816,84.48
4,20,17,411,3.54,11209,96.46


In [49]:
X_train = X_train.drop(columns=["Location"])
X_test = X_test.drop(columns=["Location"])

print("Current training features:")
print(X_train.columns.tolist())

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)

Current training features:
['Total_Area', 'Baths', 'Balcony', 'BHK', 'BHK_5_PLUS', 'Configuration', 'City', 'Location_Grouped']

Training shape: (11620, 8)
Test shape: (2905, 8)


## Location Feature Engineering

`Location` is a high-cardinality categorical feature with 7,050 unique values in the complete dataset.

Analysis on the training data showed that many locations occur only once. Therefore, directly one-hot encoding every unique location would create a very sparse feature space.

To retain useful geographic information while controlling dimensionality, two location-based features were created:

- `City`: extracted from the final comma-separated component of `Location`.
- `Location_Grouped`: retains locations that appear at least twice in the training data and maps rarer locations to `Other`.

The frequency threshold was determined using the training data only to prevent information from the test set influencing preprocessing decisions.

A minimum frequency of **2** was selected because it retains locality information for approximately **62.94% of training properties**, while higher thresholds discard substantially more locality information.

The original `Location` column was then removed from the modeling features because its information is represented by `City` and `Location_Grouped`.

In [50]:
missing_summary = pd.DataFrame({
    "Train_Missing": X_train.isna().sum(),
    "Train_Missing_%": X_train.isna().mean() * 100,
    "Test_Missing": X_test.isna().sum(),
    "Test_Missing_%": X_test.isna().mean() * 100
})

missing_summary

,Train_Missing,Train_Missing_%,Test_Missing,Test_Missing_%
Total_Area,0,0.000000,0,0.000000
Baths,0,0.000000,0,0.000000
Balcony,0,0.000000,0,0.000000
BHK,597,5.137694,130,4.475043
BHK_5_PLUS,0,0.000000,0,0.000000
Configuration,0,0.000000,0,0.000000
City,0,0.000000,0,0.000000
Location_Grouped,0,0.000000,0,0.000000


## Missing BHK Analysis

The engineered `BHK` feature contains missing values because not every property title specifies a conventional BHK configuration.

For example, `RK` and `Studio` properties do not provide a standard BHK value. Therefore, missing `BHK` values may represent a meaningful property configuration rather than random missing data.

Before selecting an imputation strategy, the relationship between missing `BHK` values and the engineered `Configuration` feature will be examined.

In [51]:
bhk_missing_analysis = pd.crosstab(
    X_train["Configuration"],
    X_train["BHK"].isna(),
    margins=True
)

bhk_missing_analysis

BHK,False,True,All
Configuration,,,
BHK,11023,0,11023
RK,0,579,579
Studio,0,9,9
Unknown,0,9,9
All,11023,597,11620


### BHK Missing-Value Decision

The `BHK` feature contains 597 missing values in the training set.

Further analysis shows that:

- All `BHK` configurations have a numeric BHK value.
- All `RK` properties have missing BHK values.
- All `Studio` properties have missing BHK values.
- The remaining missing values belong to `Unknown` configurations.

Therefore, the missing `BHK` values are structural rather than random missing data.

Median or mean imputation would introduce artificial bedroom counts for `RK`, `Studio`, and `Unknown` properties. Therefore, the raw missing values will be preserved at this stage.

The preprocessing pipeline will handle these missing numerical values later, while the `Configuration` feature retains the semantic distinction between BHK, RK, Studio, and Unknown properties.

# Preprocessing Strategy

The engineered modeling features are divided into numerical and categorical groups.

### Numerical Features

- `Total_Area`
- `Baths`
- `BHK`
- `BHK_5_PLUS`

`BHK` contains structural missing values for RK, Studio, and Unknown properties. These missing values will be handled by the numerical preprocessing pipeline rather than manually modified.

### Categorical Features

- `Balcony`
- `Configuration`
- `City`
- `Location_Grouped`

Categorical features will be handled using imputation and one-hot encoding.

### Preprocessing Principles

The preprocessing operations will be fitted only on the training data. The test set will only be transformed using the already-fitted preprocessing pipeline.

This prevents information from the test set from influencing the training process.

In [52]:
numeric_features = [
    "Total_Area",
    "Baths",
    "BHK",
    "BHK_5_PLUS"
]

categorical_features = [
    "Balcony",
    "Configuration",
    "City",
    "Location_Grouped"
]

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Total_Area', 'Baths', 'BHK', 'BHK_5_PLUS']

Categorical features:
['Balcony', 'Configuration', 'City', 'Location_Grouped']


## Preprocessing Pipeline Design

Two separate preprocessing pipelines will be created.

### Numerical Pipeline

The numerical features will use:

1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`

Median imputation is used because `BHK` contains structural missing values. The imputation value will be learned from the training data only.

### Categorical Pipeline

The categorical features will use:

1. `SimpleImputer(strategy="most_frequent")`
2. `OneHotEncoder(handle_unknown="ignore")`

`handle_unknown="ignore"` ensures that categories not observed during training do not cause transformation errors when new data is processed.

Both pipelines will later be combined using `ColumnTransformer`.

In [53]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

print("Numerical pipeline:")
print(numeric_pipeline)

print("\nCategorical pipeline:")
print(categorical_pipeline)

Numerical pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

Categorical pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))])


In [54]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

print("Numerical pipeline:")
print(numeric_pipeline)

print("\nCategorical pipeline:")
print(categorical_pipeline)

Numerical pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

Categorical pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))])


## Combined Preprocessing Transformer

The numerical and categorical pipelines will now be combined using `ColumnTransformer`.

`ColumnTransformer` allows different preprocessing operations to be applied to different feature types while maintaining a single reproducible transformation object.

The transformer will be fitted only on the training data and then used to transform both training and test data.


In [55]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessor created successfully.")
print(preprocessor)

Preprocessor created successfully.
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Total_Area', 'Baths', 'BHK', 'BHK_5_PLUS']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Balcony', 'Configuration', 'City',
                                  'Location_Grouped'])])


## Fit Preprocessor on Training Data

The preprocessing transformer will be fitted using the training data only.

This means:

- numerical imputation values are learned from training data
- scaling parameters are learned from training data
- categorical levels are learned from training data

The test data will not influence these learned parameters.

After fitting, the same transformer will be used to transform both training and test data.

In [56]:
# Fit preprocessor and transform train/test data

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)


Training processed shape: (11620, 1738)
Test processed shape: (2905, 1738)


## Processed Feature Validation

The preprocessing transformer has converted the original features into a numerical machine-learning representation.

The transformed training and test datasets contain the same number of features, which confirms that the same fitted preprocessing schema is being applied to both datasets.

The processed matrices will now be checked for missing or invalid numerical values before model training.

In [57]:
print(
    "Training processed matrix contains NaN:",
    np.isnan(X_train_processed.toarray()).any()
)

print(
    "Test processed matrix contains NaN:",
    np.isnan(X_test_processed.toarray()).any()
)

Training processed matrix contains NaN: False
Test processed matrix contains NaN: False


## Final Target Validation

Before saving the processed datasets, the target vectors are checked to ensure that their row counts and indices remain aligned with the corresponding feature datasets.

The target is kept separate from the preprocessing transformer because `Price` is the prediction target and must never be transformed as an input feature.

In [58]:
print("Training features:", X_train_processed.shape)
print("Training target:", y_train.shape)

print("Test features:", X_test_processed.shape)
print("Test target:", y_test.shape)

print("\nTraining rows aligned:", X_train_processed.shape[0] == len(y_train))
print("Test rows aligned:", X_test_processed.shape[0] == len(y_test))

Training features: (11620, 1738)
Training target: (11620,)
Test features: (2905, 1738)
Test target: (2905,)

Training rows aligned: True
Test rows aligned: True


# Preprocessing Summary

The preprocessing stage transformed the validated real-estate dataset into a reproducible machine-learning representation.

### Target

`Price` was converted into a numeric target using recognized price units. Three ambiguous price records were excluded from supervised modeling because their target values could not be reliably interpreted.

### Feature Engineering

`Property Title` was transformed into structured features:

- `BHK`
- `BHK_5_PLUS`
- `Configuration`

`Location` was transformed into:

- `City`
- `Location_Grouped`

Locations appearing fewer than two times in the training data were grouped into `Other`.

### Feature Exclusion

The following columns were excluded:

- `Name` — high cardinality and substantial overlap with `Location`
- `Property Title` — replaced by engineered features
- `Description` — target leakage risk
- `Price_per_SQFT` — target leakage risk

### Missing Values

`BHK` contained structural missing values for `RK`, `Studio`, and `Unknown` configurations. These values were not manually replaced. The numerical preprocessing pipeline handles the missing values using median imputation.

### Preprocessing Pipeline

Numerical features use:

1. Median imputation
2. Standard scaling

Categorical features use:

1. Most-frequent imputation
2. One-hot encoding with `handle_unknown="ignore"`

The combined preprocessing transformer was fitted only on the training data and then applied to both training and test data.

### Final Processed Data

- Training samples: **11,620**
- Test samples: **2,905**
- Processed features: **1,738**

Validation confirmed that the processed matrices contain no missing values and that feature rows remain aligned with their corresponding target values.

The preprocessing pipeline is now ready for model development and comparison in Notebook 05.